# 07 - Tune CTGAN (Optuna) and train the 4 final CTGANs

One CTGAN per dataset version (`original`, `ros_10k`, `smote_3k`, `smote_10k`), as planned.

**Plan**
1. **Tune** the CTGAN hyperparameters with Optuna on the `original` version (no oversampling, so a clean real hold-out split is possible).
2. The objective is a **compound score**: mean of (a) macro label consistency and (b) TSTR macro-F1 on a real hold-out that the generator never saw. Rare classes count as much as big ones.
3. **Train the 4 final CTGANs** with the best parameters and save the 4 `.pt` + 4 synthetic CSV files (+ a `_filtered` CSV with only rows whose content matches their label).

Everything uses `src/fix_pipeline.py` (log1p, discrete code columns, clipping, matched class sampling).
Tuning ideas come from the literature note: per-dataset tuning of epochs / batch size / layer widths / embedding size / learning rates, log-frequency sampling, compound objective (papers #13-16, #1-2).

*Run on a GPU (Colab) for the real run. Keep `QUICK = True` first to test the notebook (few minutes).*

In [1]:
# !pip install optuna ctgan
import sys, time, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, torch, optuna
from sklearn.model_selection import train_test_split
from src import fix_pipeline as f

print("CUDA available:", torch.cuda.is_available())

# ---- CONFIG ----
QUICK = True                 # True: tiny test run. False: real run (GPU)
SEED = 42
TUNE_ON = "original"
DATASETS = ["original", "ros_10k", "smote_3k", "smote_10k"]
DATA_DIR = ROOT / "dataset" / "UNSW_NB15" / "oversampled"
OUT_DIR = ROOT / "outputs"; OUT_DIR.mkdir(exist_ok=True)

if QUICK:
    N_TRIALS, EPOCH_MAX, PER_CLASS, FINAL_EPOCHS, TAG = 3, 50, 800, 20, "tuned_quick"
else:
    N_TRIALS, EPOCH_MAX, PER_CLASS, FINAL_EPOCHS, TAG = 20, 200, 1500, 300, "tuned"
print(dict(N_TRIALS=N_TRIALS, EPOCH_MAX=EPOCH_MAX, PER_CLASS=PER_CLASS, FINAL_EPOCHS=FINAL_EPOCHS, TAG=TAG))

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: False
{'N_TRIALS': 3, 'EPOCH_MAX': 50, 'PER_CLASS': 800, 'FINAL_EPOCHS': 20, 'TAG': 'tuned_quick'}


## 1. Data for tuning: real train / real hold-out split (the generator never sees the hold-out)

In [2]:
full = pd.read_csv(DATA_DIR / f"unsw_nb15_attacks_{TUNE_ON}.csv")
sample = full.sample(frac=1, random_state=SEED).groupby(f.LABEL).head(PER_CLASS).reset_index(drop=True)
tune_train, tune_test = train_test_split(sample, test_size=0.3, stratify=sample[f.LABEL], random_state=SEED)
print("tuning train:", tune_train.shape, "| real hold-out:", tune_test.shape)
display(pd.DataFrame({"train": tune_train[f.LABEL].value_counts(), "hold-out": tune_test[f.LABEL].value_counts()}))

tuning train: (4558, 49) | real hold-out: (1954, 49)


,train,hold-out
Attack,,
Analysis,560,240
Backdoor,560,240
DoS,560,240
Exploits,560,240
Fuzzers,560,240
Generic,560,240
Reconnaissance,560,240
Shellcode,560,240
Worms,78,34


## 2. Optuna objective (compound score) and search

In [3]:
DIMS = {"256x256": (256, 256), "256x256x256": (256, 256, 256), "512x512": (512, 512)}

def params_from(trial):
    return dict(
        epochs=trial.suggest_int("epochs", 50, EPOCH_MAX, step=50) if EPOCH_MAX >= 100 else trial.suggest_int("epochs", 10, EPOCH_MAX, step=10),
        batch_size=trial.suggest_categorical("batch_size", [100, 500, 1000]),
        pac=trial.suggest_categorical("pac", [1, 5, 10]),
        embedding_dim=trial.suggest_categorical("embedding_dim", [64, 128, 256]),
        generator_dim=DIMS[trial.suggest_categorical("generator_dim", list(DIMS))],
        discriminator_dim=DIMS[trial.suggest_categorical("discriminator_dim", list(DIMS))],
        generator_lr=trial.suggest_float("generator_lr", 1e-5, 1e-3, log=True),
        discriminator_lr=trial.suggest_float("discriminator_lr", 1e-5, 1e-3, log=True),
        discriminator_steps=trial.suggest_categorical("discriminator_steps", [1, 3, 5]),
        log_frequency=trial.suggest_categorical("log_frequency", [True, False]),
    )

def objective(trial):
    pr = params_from(trial)
    t0 = time.time()
    _, _, syn = f.train_and_sample(tune_train, seed=SEED, verbose=False, **pr)
    res, _ = f.evaluate(tune_train, tune_test, syn, seed=SEED)
    trial.set_user_attr("label_consistency_macro", res["label_consistency_macro"])
    trial.set_user_attr("tstr_macro_f1", res["tstr_macro_f1"])
    trial.set_user_attr("minutes", round((time.time() - t0) / 60, 1))
    return res["score"]

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

trials = study.trials_dataframe()
display(trials.sort_values("value", ascending=False).head(10))
trials.to_csv(OUT_DIR / f"optuna_trials_{TAG}.csv", index=False)
print("BEST score:", round(study.best_value, 4))
print("BEST params:", study.best_params)

  rows missing (generator ignored the condition): {'Fuzzers': 38, 'DoS': 84, 'Exploits': 64, 'Shellcode': 78, 'Reconnaissance': 65, 'Analysis': 18, 'Backdoor': 60, 'Generic': 101, 'Worms': 60}
  rows missing (generator ignored the condition): {'Fuzzers': 34, 'DoS': 60, 'Exploits': 48, 'Shellcode': 82, 'Reconnaissance': 52, 'Analysis': 65, 'Backdoor': 41, 'Generic': 37, 'Worms': 53}
  rows missing (generator ignored the condition): {'Fuzzers': 165, 'DoS': 46, 'Exploits': 28, 'Shellcode': 99, 'Reconnaissance': 115, 'Analysis': 130, 'Backdoor': 67, 'Worms': 44}


,number,value,datetime_start,datetime_complete,duration,params_batch_size,params_discriminator_dim,params_discriminator_lr,params_discriminator_steps,params_embedding_dim,params_epochs,params_generator_dim,params_generator_lr,params_log_frequency,params_pac,user_attrs_label_consistency_macro,user_attrs_minutes,user_attrs_tstr_macro_f1,state
0,0,0.1376,2026-11-01 13:47:02.302555,2026-11-01 13:52:12.302807,0 days 00:05:10.000252,100,256x256,0.000112,5,64,20,256x256x256,0.000041,False,1,0.1244,5.2,0.1507,COMPLETE
1,1,0.1058,2026-11-01 13:52:12.303813,2026-11-01 13:55:27.512633,0 days 00:03:15.208820,500,512x512,0.000018,5,64,20,256x256x256,0.000076,False,5,0.1109,3.3,0.1006,COMPLETE
2,2,0.0832,2026-11-01 13:55:27.512633,2026-11-01 13:57:37.151104,0 days 00:02:09.638471,500,256x256x256,0.000052,3,256,20,256x256x256,0.000454,True,1,0.1059,2.2,0.0604,COMPLETE


BEST score: 0.1376
BEST params: {'epochs': 20, 'batch_size': 100, 'pac': 1, 'embedding_dim': 64, 'generator_dim': '256x256x256', 'discriminator_dim': '256x256', 'generator_lr': 4.059611610484306e-05, 'discriminator_lr': 0.00011207606211860574, 'discriminator_steps': 5, 'log_frequency': False}


## 3. Train the 4 final CTGANs with the best parameters

In [4]:
best = dict(study.best_params)
best["generator_dim"] = DIMS[best.pop("generator_dim")]
best["discriminator_dim"] = DIMS[best.pop("discriminator_dim")]
best["epochs"] = FINAL_EPOCHS            # final training can be longer than the tuning trials
print("final parameters:", best)
json.dump({k: (list(v) if isinstance(v, tuple) else v) for k, v in best.items()},
          open(OUT_DIR / f"best_params_{TAG}.json", "w"), indent=1)

rows, cons = [], {}
for name in DATASETS:
    real = pd.read_csv(DATA_DIR / f"unsw_nb15_attacks_{name}.csv")
    if QUICK: real = real.sample(frac=1, random_state=SEED).groupby(f.LABEL).head(PER_CLASS).reset_index(drop=True)
    t0 = time.time()
    model, p, syn = f.train_and_sample(real, seed=SEED, verbose=False, **best)
    model.save(str(OUT_DIR / f"ctgan_unsw_nb15_{name}_{TAG}.pt"))
    syn.to_csv(OUT_DIR / f"synthetic_ctgan_unsw_nb15_{name}_{TAG}.csv", index=False)
    ok, per_class = f.label_consistency(real, syn, seed=SEED)
    syn[ok.to_numpy()].to_csv(OUT_DIR / f"synthetic_ctgan_unsw_nb15_{name}_{TAG}_filtered.csv", index=False)
    cons[name] = per_class
    num = syn.select_dtypes("number"); v = f.cat_validity(real, syn, p)
    rows.append({"dataset": name, "rows": len(syn), "minutes": round((time.time() - t0) / 60, 1),
                 "negative_cells": int((num < 0).sum().sum()), "NaN_cells": int(syn.isna().sum().sum()),
                 "cat_validity_min": v.min(), "label_consistency_macro": round(float(per_class.mean()), 3),
                 "rows_kept_after_filter": int(ok.sum())})
    print(f"[{name}] done")

final parameters: {'epochs': 20, 'batch_size': 100, 'pac': 1, 'embedding_dim': 64, 'generator_lr': 4.059611610484306e-05, 'discriminator_lr': 0.00011207606211860574, 'discriminator_steps': 5, 'log_frequency': False, 'generator_dim': (256, 256, 256), 'discriminator_dim': (256, 256)}
  rows missing (generator ignored the condition): {'Fuzzers': 84, 'Exploits': 80, 'Reconnaissance': 70, 'Generic': 156, 'Shellcode': 91, 'Backdoor': 126, 'Analysis': 17, 'DoS': 109, 'Worms': 97}
[original] done
  rows missing (generator ignored the condition): {'Analysis': 33, 'Exploits': 147, 'Worms': 79, 'Fuzzers': 110, 'Backdoor': 140, 'Reconnaissance': 102, 'Shellcode': 135, 'Generic': 198, 'DoS': 146}
[ros_10k] done
  rows missing (generator ignored the condition): {'Shellcode': 128, 'DoS': 160, 'Exploits': 157, 'Analysis': 48, 'Worms': 54, 'Fuzzers': 81, 'Generic': 187, 'Reconnaissance': 81, 'Backdoor': 105}
[smote_3k] done
  rows missing (generator ignored the condition): {'Analysis': 34, 'Exploits': 

## 4. Results

In [5]:
summary = pd.DataFrame(rows).set_index("dataset")
display(summary)
print("Label consistency per attack class:")
display(pd.DataFrame(cons).round(3))
summary.to_csv(OUT_DIR / f"ctgan_{TAG}_summary.csv")
pd.DataFrame(cons).to_csv(OUT_DIR / f"ctgan_{TAG}_label_consistency.csv")

,rows,minutes,negative_cells,NaN_cells,cat_validity_min,label_consistency_macro,rows_kept_after_filter
dataset,,,,,,,
original,5682,6.3,0,0,1.0,0.119,743
ros_10k,6110,7.2,0,0,1.0,0.118,703
smote_3k,6199,7.7,0,0,1.0,0.115,686
smote_10k,6145,7.9,0,0,1.0,0.121,742


Label consistency per attack class:


,original,ros_10k,smote_3k,smote_10k
Attack,,,,
Analysis,0.003,0.008,0.001,0.009
Backdoor,0.221,0.198,0.161,0.209
DoS,0.460,0.460,0.416,0.433
Exploits,0.124,0.144,0.123,0.166
Fuzzers,0.091,0.087,0.063,0.055
Generic,0.040,0.058,0.067,0.052
Reconnaissance,0.026,0.044,0.063,0.027
Shellcode,0.106,0.060,0.132,0.126
Worms,0.000,0.007,0.011,0.013


## How to read
* **Tuning table:** the best trials show which settings help. Compare `label_consistency_macro` and `tstr_macro_f1` of the best trial with an average trial. A flat table (all scores similar) means tuning does not move the metric and the limit is elsewhere (class overlap in the data, or too little data for rare classes).
* **Results:** `negative_cells` and `NaN_cells` must be 0, `cat_validity_min` close to 1. The label consistency of the real hold-out (the ceiling) is not shown here; add it when reporting.
* Final models are trained on the full file of each version. The generator train / hold-out question for the project split (D_G, D_S, D_H) is still open with the Team Leader.
* `QUICK = True` results only test that the notebook runs. Do not use them to judge quality.